## 1. Definizione Percorsi e Inventario Artefatti

In [1]:
import pandas as pd
import numpy as np
import json
import logging
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Tuple
import os

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Definizione percorsi
PROJECT_ROOT = Path('..')
DATA_RAW = PROJECT_ROOT / 'datasets' / 'raw' / 'ml-latest-small'
DATA_PROCESSED = PROJECT_ROOT / 'datasets' / 'processed'
CHECKPOINTS_DIR = DATA_PROCESSED / 'checkpoints'
NORMALIZED_DIR = DATA_PROCESSED / 'normalized'

print("\n" + "="*80)
print("SETUP PERCORSI")
print("="*80)

# Valida che directory esistano
assert DATA_PROCESSED.exists(), f"DATA_PROCESSED non trovata: {DATA_PROCESSED}"
assert CHECKPOINTS_DIR.exists(), f"CHECKPOINTS_DIR non trovata: {CHECKPOINTS_DIR}"
assert NORMALIZED_DIR.exists(), f"NORMALIZED_DIR non trovata: {NORMALIZED_DIR}"

print(f"✓ Project root: {PROJECT_ROOT}")
print(f"✓ Data processed: {DATA_PROCESSED}")
print(f"✓ Checkpoints: {CHECKPOINTS_DIR}")
print(f"✓ Normalized: {NORMALIZED_DIR}")

# Inventory struttura
inventory_structure = {
    'Raw Checkpoints': {
        'location': str(CHECKPOINTS_DIR),
        'purpose': 'Raw JSON da API (non processati, reference)',
        'files': ['tmdb_raw_*.jsonl', 'fetch_stats_*.json', 'tmdb_parsed_*.csv'],
        'use_case': ['Audit completo', 'Ripresa senza refetch API', 'Accesso a tutti 60+ campi'],
    },
    'Normalized Tables': {
        'location': str(NORMALIZED_DIR),
        'purpose': 'Dati strutturati in tabelle separate (M:N)',
        'files': ['movies_enriched_*.csv', 'cast_*.csv', 'keywords_*.csv', 'crew_*.csv', 'production_companies_*.csv', 'data_dictionary.md'],
        'use_case': ['Analisi relazionale', 'Filtering avanzato', 'Preserved data'],
    },
    'Final Output': {
        'location': str(DATA_PROCESSED),
        'purpose': 'Dataset finale merged con prefissi ml_/tmdb_',
        'files': ['dataset_enriched_movielens_tmdb.csv', 'dataset_enriched_movielens_tmdb.parquet'],
        'use_case': ['EDA', 'Ranking test', 'Machine learning'],
    }
}

print("\n" + "─"*80)
print("INVENTARIO STRUTTURA ARTEFATTI")
print("─"*80)
for stage, info in inventory_structure.items():
    print(f"\n📦 {stage}")
    print(f"  Location: {info['location']}")
    print(f"  Purpose: {info['purpose']}")
    print(f"  Files: {', '.join(info['files'])}")
    print(f"  Use: {', '.join(info['use_case'])}")


SETUP PERCORSI
✓ Project root: ..
✓ Data processed: ../datasets/processed
✓ Checkpoints: ../datasets/processed/checkpoints
✓ Normalized: ../datasets/processed/normalized

────────────────────────────────────────────────────────────────────────────────
INVENTARIO STRUTTURA ARTEFATTI
────────────────────────────────────────────────────────────────────────────────

📦 Raw Checkpoints
  Location: ../datasets/processed/checkpoints
  Purpose: Raw JSON da API (non processati, reference)
  Files: tmdb_raw_*.jsonl, fetch_stats_*.json, tmdb_parsed_*.csv
  Use: Audit completo, Ripresa senza refetch API, Accesso a tutti 60+ campi

📦 Normalized Tables
  Location: ../datasets/processed/normalized
  Purpose: Dati strutturati in tabelle separate (M:N)
  Files: movies_enriched_*.csv, cast_*.csv, keywords_*.csv, crew_*.csv, production_companies_*.csv, data_dictionary.md
  Use: Analisi relazionale, Filtering avanzato, Preserved data

📦 Final Output
  Location: ../datasets/processed
  Purpose: Dataset fin

## 2. Discovery Automatico File e Inventario Timestamp

In [2]:
print("\n" + "="*80)
print("DISCOVERY AUTOMATICO FILE")
print("="*80)

# Scansione directory
checkpoint_files = sorted(CHECKPOINTS_DIR.glob('tmdb_raw_*.jsonl'), key=os.path.getmtime, reverse=True)
normalized_files = sorted(NORMALIZED_DIR.glob('*.csv'), key=os.path.getmtime, reverse=True)
final_outputs = sorted(DATA_PROCESSED.glob('dataset_enriched_movielens_tmdb*'), key=os.path.getmtime, reverse=True)

print(f"\n📁 Checkpoints trovati: {len(checkpoint_files)}")
for f in checkpoint_files[:3]:  # Ultimi 3
    size_mb = f.stat().st_size / (1024*1024)
    mtime = datetime.fromtimestamp(f.stat().st_mtime).strftime('%Y-%m-%d %H:%M:%S')
    print(f"  - {f.name} ({size_mb:.2f} MB, mod: {mtime})")

print(f"\n📁 Tabelle normalizzate trovate: {len(normalized_files)}")
for f in normalized_files[:5]:  # Ultimi 5
    size_kb = f.stat().st_size / 1024
    mtime = datetime.fromtimestamp(f.stat().st_mtime).strftime('%Y-%m-%d %H:%M:%S')
    print(f"  - {f.name} ({size_kb:.1f} KB, mod: {mtime})")

print(f"\n📁 Output finali: {len(final_outputs)}")
for f in final_outputs:
    size_mb = f.stat().st_size / (1024*1024)
    mtime = datetime.fromtimestamp(f.stat().st_mtime).strftime('%Y-%m-%d %H:%M:%S')
    print(f"  - {f.name} ({size_mb:.2f} MB, mod: {mtime})")

# Identifica l'ultima run
if checkpoint_files:
    latest_raw = checkpoint_files[0]
    # Estrai timestamp dal filename
    timestamp = latest_raw.stem.replace('tmdb_raw_', '')
    print(f"\n🔄 Latest run timestamp: {timestamp}")
else:
    timestamp = None
    print("\n⚠ Nessun checkpoint trovato!")


DISCOVERY AUTOMATICO FILE

📁 Checkpoints trovati: 2
  - tmdb_raw_20260729_154149.jsonl (3.49 MB, mod: 2026-07-29 15:41:49)
  - tmdb_raw_20260729_153739.jsonl (3.49 MB, mod: 2026-07-29 15:37:39)

📁 Tabelle normalizzate trovate: 8
  - production_companies_20260729_154149.csv (3.9 KB, mod: 2026-07-29 15:41:49)
  - crew_20260729_154149.csv (13.0 KB, mod: 2026-07-29 15:41:49)
  - keywords_20260729_154149.csv (23.4 KB, mod: 2026-07-29 15:41:49)
  - cast_20260729_154149.csv (176.9 KB, mod: 2026-07-29 15:41:49)
  - production_companies_20260729_153739.csv (3.9 KB, mod: 2026-07-29 15:37:59)

📁 Output finali: 2
  - dataset_enriched_movielens_tmdb.parquet (0.16 MB, mod: 2026-07-29 15:41:49)
  - dataset_enriched_movielens_tmdb.csv (0.17 MB, mod: 2026-07-29 15:41:49)

🔄 Latest run timestamp: 20260729_154149


## 3. lettura dati

In [7]:
try:
    # Carica DF dati
    movies_files = list(DATA_PROCESSED.glob('dataset_enriched_movielens_tmdb.csv'))
    assert len(movies_files) > 0, "Nessun movies_enriched trovato"
    df_test_enriched = pd.read_csv(movies_files[0])
    assert len(df_test_enriched) > 0, "Dataset merged vuoto"
    print("✓ Test 2: Dataset merged caricato con successo")
    
    # Test 1: verifico unicità DF
    assert df_test_enriched['movieId'].nunique() == len(df_test_enriched), "movieId non unique"
    print("✓ Test 1: movieId unique")
    
    print("\nDati caricati pronti all'untilizzo")
    print(df_test_enriched.head())
    
except Exception as e:
    print(f"\n✗ ERRORE lettura dati: {e}")

✓ Test 2: Dataset merged caricato con successo
✓ Test 1: movieId unique

Dati caricati pronti all'untilizzo
   movieId  imdbId  tmdbId                   canonical_title  \
0      356  109830      13               Forrest Gump (1994)   
1      318  111161     278  Shawshank Redemption, The (1994)   
2      296  110912     680               Pulp Fiction (1994)   
3      593  102926     274  Silence of the Lambs, The (1991)   
4     2571  133093     603                Matrix, The (1999)   

                           ml_title                tmdb_title  \
0               Forrest Gump (1994)              Forrest Gump   
1  Shawshank Redemption, The (1994)  The Shawshank Redemption   
2               Pulp Fiction (1994)              Pulp Fiction   
3  Silence of the Lambs, The (1991)  The Silence of the Lambs   
4                Matrix, The (1999)                The Matrix   

        tmdb_original_title release_date  release_year  \
0              Forrest Gump   1994-06-23          1994   


## 4. selezione colonne